# RAG Assignment

### Importing libraries

In [1]:
!pip install -qU langchain langchain-community

!pip install -qU opentelemetry-api opentelemetry-sdk opentelemetry-exporter-otlp chromadb


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 112.7/112.7 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 81.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 56.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 508.7/508.7 kB 30.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 169.8/169.8 kB 13.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 4.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.35.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.25.1 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
google-colab 1.0.0 requires jupyter-server==2.14.0, but you have jupyter-server 2.12.5 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.2, but you have 

### Loading data into the environment

In [2]:
from pathlib import Path

from langchain_community.document_loaders import TextLoader, DirectoryLoader
from langchain_core.documents import Document


# Loading dataset -input documents

class DocumentLoader:

    def load_file(self, path: str | Path) -> Document:
        loader = TextLoader(str(path), encoding="utf-8")   # reads a single .txt file
        return loader.load()[0]                             # .load() returns a list; grab the one document

    def load_folder(self, folder: str | Path, recursive: bool = False) -> list[Document]:
        loader = DirectoryLoader(
            str(folder),                                    # root folder to scan
            glob          = "**/*.txt" if recursive else "*.txt",  # file pattern: deep scan or top-level only
            loader_cls    = TextLoader,                     # which loader to use for each matched file
            loader_kwargs = {"encoding": "utf-8"},          # extra args forwarded to TextLoader
            show_progress = True,                           # print a progress bar while loading
        )
        docs = loader.load()                                # loads all matched files into Document objects
        for doc in docs:
            print(f"  [OK]   {Path(doc.metadata['source']).name} — {len(doc.page_content)} chars")
        return docs

    def inspect(self, docs: list[Document]) -> None:
        print(f"\n{'─'*55}")
        print(f"Loaded {len(docs)} document(s)\n")
        for i, doc in enumerate(docs):
            print(f"  [{i}] source : {doc.metadata.get('source')}")
            print(f"       chars  : {len(doc.page_content)}")
            print(f"       preview: {doc.page_content[:80].replace(chr(10), ' ')!r}")
            print()
        print(f"{'─'*55}\n")


if __name__ == "__main__":

    loader = DocumentLoader()
    docs   = loader.load_folder("/kaggle/input/datasets/oyeyemidareazeez/input-docs/")
    loader.inspect(docs)


100%|██████████| 6/6 [00:00<00:00, 242.79it/s]

  [OK]   large_language_models.txt — 16611 chars
  [OK]   climate_change.txt — 4159 chars
  [OK]   machine_learning.txt — 12801 chars
  [OK]   ai_history.txt — 3854 chars
  [OK]   space_exploration.txt — 4031 chars
  [OK]   quantum_computing.txt — 13401 chars

───────────────────────────────────────────────────────
Loaded 6 document(s)

  [0] source : /kaggle/input/datasets/oyeyemidareazeez/input-docs/large_language_models.txt
       chars  : 16611
       preview: 'Large Language Models: Architecture, Training, and Impact  Large language models'

  [1] source : /kaggle/input/datasets/oyeyemidareazeez/input-docs/climate_change.txt
       chars  : 4159
       preview: 'Climate change refers to long-term shifts in global temperatures and weather pat'

  [2] source : /kaggle/input/datasets/oyeyemidareazeez/input-docs/machine_learning.txt
       chars  : 12801
       preview: 'Machine Learning: Foundations, Algorithms, and Applications  Machine learning is'

  [3] source : /kaggle/input/dat

### Chunking Data

In [3]:
# Chunker 
from langchain_text_splitters import CharacterTextSplitter

class FixedChunker:

    def __init__(self, chunk_size: int = 500, overlap: int = 50):
        self.chunk_size = chunk_size
        self.overlap    = overlap
        self.splitter   = CharacterTextSplitter(
            chunk_size    = chunk_size,   # max characters per chunk
            chunk_overlap = overlap,      # how many chars the next chunk re-uses from the previous one
            separator     = "",           # pure character-level, no semantic splitting
        )

    def chunk_many(self, docs: list[Document]) -> list[Document]:
        all_chunks = []
        for doc in docs:
            chunks = self.splitter.split_documents([doc])   # splits one Document into many
            for i, chunk in enumerate(chunks):
                chunk.metadata["chunk_id"] = i              # tag each chunk with its position index
            all_chunks.extend(chunks)                       # collect across all documents
            print(f"  [chunked] {doc.metadata.get('source')} → {len(chunks)} chunks")
        return all_chunks

    def inspect(self, chunks: list[Document]) -> None:
        print(f"\n{'─'*55}")
        print(f"Total chunks : {len(chunks)}")
        print(f"Chunk size   : {self.chunk_size} chars")
        print(f"Overlap      : {self.overlap} chars\n")
        for i, chunk in enumerate(chunks[:10]):
            print(f"  [{i}] source   : {chunk.metadata.get('source')}")
            print(f"       chunk_id : {chunk.metadata.get('chunk_id')}")
            print(f"       chars    : {len(chunk.page_content)}")
            print(f"       preview  : {chunk.page_content[:80].replace(chr(10), ' ')!r}")
            print()
        if len(chunks) > 10:
            print(f"  ... and {len(chunks) - 5} more chunks")
        print(f"{'─'*55}\n")


# Main

if __name__ == "__main__":

    loader  = DocumentLoader()
    chunker = FixedChunker(chunk_size=500, overlap=50)
    chunks = chunker.chunk_many(docs)
    chunker.inspect(chunks)

  [chunked] /kaggle/input/datasets/oyeyemidareazeez/input-docs/large_language_models.txt → 37 chunks
  [chunked] /kaggle/input/datasets/oyeyemidareazeez/input-docs/climate_change.txt → 10 chunks
  [chunked] /kaggle/input/datasets/oyeyemidareazeez/input-docs/machine_learning.txt → 29 chunks
  [chunked] /kaggle/input/datasets/oyeyemidareazeez/input-docs/ai_history.txt → 9 chunks
  [chunked] /kaggle/input/datasets/oyeyemidareazeez/input-docs/space_exploration.txt → 9 chunks
  [chunked] /kaggle/input/datasets/oyeyemidareazeez/input-docs/quantum_computing.txt → 30 chunks

───────────────────────────────────────────────────────
Total chunks : 124
Chunk size   : 500 chars
Overlap      : 50 chars

  [0] source   : /kaggle/input/datasets/oyeyemidareazeez/input-docs/large_language_models.txt
       chunk_id : 0
       chars    : 500
       preview  : 'Large Language Models: Architecture, Training, and Impact  Large language models'

  [1] source   : /kaggle/input/datasets/oyeyemidareazeez/input-

### Storing Embeddings using Chroma

In [4]:
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma


# Embedder
class Embedder:

    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        print(f"  [loading model] {model_name} ...")
        self.model_name = model_name
        self.embeddings = HuggingFaceEmbeddings(model_name=model_name)


# Vector store 

class VectorStore:

    def __init__(self, embedder: Embedder, collection_name: str = "rag", persist_dir: str = "chroma_db"):
        self.collection_name = collection_name
        self.db = Chroma(
            collection_name    = collection_name,       # name of the table inside ChromaDB
            embedding_function = embedder.embeddings,   # used to embed queries at search time
            persist_directory  = persist_dir,           # folder where ChromaDB saves data to disk
        )
        print(f"  [chroma] collection '{collection_name}' — {self.db._collection.count()} docs stored")

    def store(self, chunks: list[Document]) -> None:
        self.db.add_documents(chunks)                   # embeds each chunk and stores vector + text
        print(f"  [stored] {len(chunks)} chunks → total in db: {self.db._collection.count()}")

    def inspect(self) -> None:
        count = self.db._collection.count()
        print(f"\n{'─'*55}")
        print(f"Collection   : {self.collection_name}")
        print(f"Total chunks : {count}")
        if count > 0:
            sample = self.db._collection.peek(3)        # fetch first 3 entries without a query
            print(f"\nFirst {len(sample['ids'])} entries:\n")
            for i, (id_, doc) in enumerate(zip(sample["ids"], sample["documents"])):
                print(f"  [{i}] id      : {id_}")
                print(f"       preview : {doc[:80].replace(chr(10), ' ')!r}")
                print()
        print(f"{'─'*55}\n")


# Main

if __name__ == "__main__":

    # embed & store
    embedder = Embedder()
    store    = VectorStore(embedder=embedder, collection_name="rag", persist_dir="chroma_db")
    store.store(chunks)
    store.inspect()


  [loading model] all-MiniLM-L6-v2 ...


/tmp/ipykernel_55/1420687046.py:11: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  self.embeddings = HuggingFaceEmbeddings(model_name=model_name)


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

/tmp/ipykernel_55/1420687046.py:20: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  self.db = Chroma(


  [chroma] collection 'rag' — 0 docs stored
  [stored] 124 chunks → total in db: 124

───────────────────────────────────────────────────────
Collection   : rag
Total chunks : 124

First 3 entries:

  [0] id      : 6d32dea7-5b13-457e-8675-ac5aa1c6aea3
       preview : 'Large Language Models: Architecture, Training, and Impact  Large language models'

  [1] id      : 6a3f9ac4-7685-4965-9288-5d198dcdba6d
       preview : 'on broad text data can be adapted to perform question answering, summarization, '

  [2] id      : 51cd4a5f-d267-40f7-a7d3-330fd20b6f2b
       preview : 'model to relate every token in a sequence to every other token directly, regardl'

───────────────────────────────────────────────────────



### Retriever Class

In [18]:
class Retriever:

    def __init__(self, store: VectorStore, top_k: int = 3):
        self.store = store
        self.top_k = top_k

    def retrieve(self, query: str) -> list[dict]:
        # LangChain's Chroma embeds the query and searches in one call
        results = self.store.db.similarity_search_with_score(query, k=self.top_k)
        # returns list of (Document, distance) pairs, sorted best-first

        hits = []
        for doc, distance in results:
            hits.append({
                "content":  doc.page_content,
                "metadata": doc.metadata,
                "score":    1 / (1 + distance),   # cosine distance is 0=identical → flip to similarity
            })

        return hits

    def inspect(self, query: str, hits: list[dict]) -> None:
        print(f"\n{'─'*55}")
        print(f"Query : {query!r}")
        print(f"Top-{self.top_k} results\n")
        for i, hit in enumerate(hits):
            print(f"  [{i}] score    : {hit['score']}")
            print(f"       source   : {hit['metadata'].get('source')}")
            print(f"       chunk_id : {hit['metadata'].get('chunk_id')}")
            print(f"       preview  : {hit['content'][:100].replace(chr(10), ' ')!r}")
            print()
        print(f"{'─'*55}\n")


# Main

if __name__ == "__main__":

    retriever = Retriever(store=store, top_k=3)

    query = "What is RAG?"
    hits  = retriever.retrieve(query)
    retriever.inspect(query, hits)



───────────────────────────────────────────────────────
Query : 'What is RAG?'
Top-3 results

  [0] score    : 0.44927834139401374
       source   : /kaggle/input/datasets/oyeyemidareazeez/input-docs/large_language_models.txt
       chunk_id : 21
       preview  : 'Models, a standardized evaluation framework).  Retrieval-Augmented Generation (RAG) extends LLMs by '

  [1] score    : 0.44927834139401374
       source   : /kaggle/input/datasets/oyeyemidareazeez/input-docs/large_language_models.txt
       chunk_id : 21
       preview  : 'Models, a standardized evaluation framework).  Retrieval-Augmented Generation (RAG) extends LLMs by '

  [2] score    : 0.44927834139401374
       source   : /kaggle/input/datasets/oyeyemidareazeez/input-docs/large_language_models.txt
       chunk_id : 21
       preview  : 'Models, a standardized evaluation framework).  Retrieval-Augmented Generation (RAG) extends LLMs by '

───────────────────────────────────────────────────────



### Prompt Template

In [19]:
from langchain_core.prompts import PromptTemplate

# Prompt builder

class PromptBuilder:

    def __init__(self, max_context_chars: int = 2000):
        self.max_context_chars = max_context_chars
        self.template = PromptTemplate.from_template(   # {context} and {question} are the placeholders
            "Use ONLY the context below to answer the question.\n"
            "If the answer is not in the context, say 'I don't know'.\n\n"
            "Context:\n{context}\n\n"
            "Question: {question}\n"
            "Answer:"
        )

    def build(self, query: str, hits: list[dict]) -> str:
        context = self._build_context(hits)
        return self.template.format(context=context, question=query)  # fills placeholders → returns plain string

    def inspect(self, prompt: str) -> None:
        print(f"\n{'─'*55}")
        print(f"Prompt ({len(prompt)} chars)\n")
        print(prompt)
        print(f"{'─'*55}\n")

    # ── Private ───────────────────────────────────────────────────────────────

    def _build_context(self, hits: list[dict]) -> str:
        # Concatenate retrieved chunks into a single context string.
        # Chunks are added in order of relevance (best score first).
        # We stop early if adding the next chunk would exceed max_context_chars,
        # so the prompt never gets too long for the LLM.
        # Each chunk is labeled with its rank and source file for traceability.
        chunks = []
        total  = 0

        for i, hit in enumerate(hits):
            content = hit["content"].strip()
            source  = hit["metadata"].get("source", "?")

            if total + len(content) > self.max_context_chars:   # budget check
                break

            chunks.append(f"[{i+1}] (source: {source})\n{content}")
            total += len(content)

        return "\n\n".join(chunks)


# Main

if __name__ == "__main__":

    builder   = PromptBuilder(max_context_chars=2000)

    query  = "What is RAG?"
    hits   = retriever.retrieve(query)
    prompt = builder.build(query, hits)
    builder.inspect(prompt)



───────────────────────────────────────────────────────
Prompt (1924 chars)

Use ONLY the context below to answer the question.
If the answer is not in the context, say 'I don't know'.

Context:
[1] (source: /kaggle/input/datasets/oyeyemidareazeez/input-docs/large_language_models.txt)
Models, a standardized evaluation framework).

Retrieval-Augmented Generation (RAG) extends LLMs by retrieving relevant documents from an external knowledge base at inference time. The retrieved text is injected into the model's context window alongside the user's query, allowing the model to answer questions about information that was not in its training data or that has changed since training. RAG reduces hallucination and allows factual grounding with citations.

CONTEXT LENGTH AND MEMORY

T

[2] (source: /kaggle/input/datasets/oyeyemidareazeez/input-docs/large_language_models.txt)
Models, a standardized evaluation framework).

Retrieval-Augmented Generation (RAG) extends LLMs by retrieving relevant d

### LLM Class

In [20]:
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline as hf_pipeline
from langchain_community.llms import HuggingFacePipeline

# LLM

class LLMClient:

    def __init__(self, model_name: str = "Qwen/Qwen3-1.7B", temperature: float = 0.0):
        print(f"  [loading model] {model_name} ...")
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        model          = AutoModelForCausalLM.from_pretrained(
            model_name,
            torch_dtype = "auto",   # picks float16/bfloat16 automatically based on the GPU
            device_map  = "auto",   # spreads model layers across available GPUs/CPU
        )
        pipe = hf_pipeline(
            "text-generation",
            model            = model,
            tokenizer        = self.tokenizer,
            max_new_tokens   = 512,                         # hard cap on generated tokens
            do_sample        = temperature > 0,             # True = random sampling, False = greedy
            temperature      = temperature if temperature > 0 else None,  # controls randomness; None = greedy
            return_full_text = False,                       # return only new tokens, not the prompt
            eos_token_id     = self.tokenizer.eos_token_id, # stop generating when the model emits EOS
        )
        self.llm = HuggingFacePipeline(pipeline=pipe)      # wraps the pipeline as a LangChain Runnable

    def generate(self, prompt: str) -> str:
        # Qwen3 requires chat-template format; raw prompts cause repetition loops
        formatted = self.tokenizer.apply_chat_template(
            [{"role": "user", "content": prompt}],
            tokenize              = False,          # return a string, not token ids
            add_generation_prompt = True,           # appends the assistant turn opener so the model knows to reply
            enable_thinking       = False,          # disables Qwen3's chain-of-thought scratchpad
        )
        answer = self.llm.invoke(formatted)         # runs the pipeline and returns only the new tokens
        return answer.strip()

    def inspect(self, query: str, answer: str) -> None:
        print(f"\n{'─'*55}")
        print(f"Query  : {query!r}")
        print(f"Answer : {answer}")
        print(f"{'─'*55}\n")


# Main

if __name__ == "__main__":

    embedder  = Embedder()
    store     = VectorStore(embedder=embedder, collection_name="rag", persist_dir="chroma_db")
    retriever = Retriever(store=store, top_k=3)
    builder   = PromptBuilder(max_context_chars=2000)
    llm       = LLMClient(model_name="Qwen/Qwen3-1.7B", temperature=0.0)

    query  = "What did john mcCarthy do in 1956?"
    hits   = retriever.retrieve(query)
    prompt = builder.build(query, hits)
    answer = llm.generate(prompt)
    llm.inspect(query, answer)


  [loading model] all-MiniLM-L6-v2 ...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  [chroma] collection 'rag' — 496 docs stored
  [loading model] Qwen/Qwen3-1.7B ...


Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



───────────────────────────────────────────────────────
Query  : 'What did john mcCarthy do in 1956?'
Answer : John McCarthy coined the term "artificial intelligence" in 1956 at the Dartmouth Conference.
───────────────────────────────────────────────────────



### Main Program

In [21]:
from langchain_core.runnables import RunnablePassthrough, RunnableLambda

# Config 

DATA_DIR    = "/kaggle/input/datasets/oyeyemidareazeez/input-docs/"
PERSIST_DIR = "chroma_db"
COLLECTION  = "rag"
CHUNK_SIZE  = 500
OVERLAP     = 50
TOP_K       = 3
MIN_SCORE   = 0.5
QUESTION    = "What is RAG?"


# Index 

def index() -> None:

    print("\n── Step 1  Load ──────────────────────────────────")
    loader = DocumentLoader()
    docs   = loader.load_folder(DATA_DIR)
    loader.inspect(docs)

    print("── Step 2  Chunk ─────────────────────────────────")
    chunker = FixedChunker(chunk_size=CHUNK_SIZE, overlap=OVERLAP)
    chunks  = chunker.chunk_many(docs)
    chunker.inspect(chunks)

    print("── Step 3  Embed & Store ─────────────────────────")
    embedder = Embedder()
    store    = VectorStore(embedder=embedder, collection_name=COLLECTION, persist_dir=PERSIST_DIR)
    store.store(chunks)
    store.inspect()

    print("── Indexing done ─────────────────────────────────\n")


# Query 

def query(question: str) -> str:

    print("\n── Step 4-6  RAG Chain ───────────────────────────")

    # components
    embedder  = Embedder()
    store     = VectorStore(embedder=embedder, collection_name=COLLECTION, persist_dir=PERSIST_DIR)
    retriever = Retriever(store=store, top_k=TOP_K)
    builder   = PromptBuilder(max_context_chars=2000)
    llm       = LLMClient(model_name="Qwen/Qwen3-1.7B", temperature=0.0)

    lc_retriever = store.db.as_retriever(search_kwargs={"k": TOP_K})
    # as_retriever() turns the Chroma vector store into a LangChain Runnable retriever
    # search_kwargs={"k": TOP_K} → how many chunks to return per query

    def format_docs(docs):
        # joins the retrieved Document objects into a single numbered string for the prompt
        return "\n\n".join(
            f"[{i+1}] (source: {doc.metadata.get('source', '?')})\n{doc.page_content}"
            for i, doc in enumerate(docs)
        )

    # ── LCEL chain: retriever | prompt | llm ──────────────────────────────────
    chain = (
        {"context": lc_retriever | format_docs, "question": RunnablePassthrough()}
        # "context"  → runs the retriever then format_docs on the question
        # "question" → RunnablePassthrough() forwards the question unchanged
        | builder.template                                  # fills {context} and {question} into the prompt
        | RunnableLambda(lambda x: llm.generate(x.text))   # x is StringPromptValue; .text extracts the string
    )

    answer = chain.invoke(question)

    # show hits & sources via the manual retriever for inspection
    hits = retriever.retrieve(question)
    retriever.inspect(question, hits)

    print("── Sources ───────────────────────────────────────")
    seen = []
    for hit in hits:
        if hit["score"] < MIN_SCORE:
            continue
        source   = hit["metadata"].get("source")
        chunk_id = hit["metadata"].get("chunk_id")
        score    = hit["score"]
        entry    = f"  {source}  (chunk {chunk_id},  score {score})"
        if entry not in seen:
            seen.append(entry)
            print(entry)
    print()

    llm.inspect(question, answer)
    return answer


# Main 

if __name__ == "__main__":

    index()
    query(QUESTION)



── Step 1  Load ──────────────────────────────────



100%|██████████| 6/6 [00:00<00:00, 839.67it/s]

  [OK]   large_language_models.txt — 16611 chars
  [OK]   climate_change.txt — 4159 chars
  [OK]   machine_learning.txt — 12801 chars
  [OK]   ai_history.txt — 3854 chars
  [OK]   space_exploration.txt — 4031 chars
  [OK]   quantum_computing.txt — 13401 chars

───────────────────────────────────────────────────────
Loaded 6 document(s)

  [0] source : /kaggle/input/datasets/oyeyemidareazeez/input-docs/large_language_models.txt
       chars  : 16611
       preview: 'Large Language Models: Architecture, Training, and Impact  Large language models'

  [1] source : /kaggle/input/datasets/oyeyemidareazeez/input-docs/climate_change.txt
       chars  : 4159
       preview: 'Climate change refers to long-term shifts in global temperatures and weather pat'

  [2] source : /kaggle/input/datasets/oyeyemidareazeez/input-docs/machine_learning.txt
       chars  : 12801
       preview: 'Machine Learning: Foundations, Algorithms, and Applications  Machine learning is'

  [3] source : /kaggle/input/dat

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


...

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  [chroma] collection 'rag' — 620 docs stored
  [loading model] Qwen/Qwen3-1.7B ...


Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



───────────────────────────────────────────────────────
Query : 'What is RAG?'
Top-3 results

  [0] score    : 0.44927834139401374
       source   : /kaggle/input/datasets/oyeyemidareazeez/input-docs/large_language_models.txt
       chunk_id : 21
       preview  : 'Models, a standardized evaluation framework).  Retrieval-Augmented Generation (RAG) extends LLMs by '

  [1] score    : 0.44927834139401374
       source   : /kaggle/input/datasets/oyeyemidareazeez/input-docs/large_language_models.txt
       chunk_id : 21
       preview  : 'Models, a standardized evaluation framework).  Retrieval-Augmented Generation (RAG) extends LLMs by '

  [2] score    : 0.44927834139401374
       source   : /kaggle/input/datasets/oyeyemidareazeez/input-docs/large_language_models.txt
       chunk_id : 21
       preview  : 'Models, a standardized evaluation framework).  Retrieval-Augmented Generation (RAG) extends LLMs by '

───────────────────────────────────────────────────────

── Sources ────────────

In [41]:
import os
from pathlib import Path
import pandas as pd
import gradio as gr

# setup function for the RAG based on class .py codes

# Loads all reusable RAG components once at startup.  Builds the Chroma DB if it is empty, otherwise reuses it.
def setup_rag():
    
    print("\n RAG STARTUP")

    # Load embedding model once
    embedder = Embedder()

    # Connect to the vector store once
    store = VectorStore(
        embedder=embedder,
        collection_name=COLLECTION,
        persist_dir=PERSIST_DIR
    )

    # Checking if ChromaDB is empty, index documents once
    if store.db._collection.count() == 0:
        print("\n[INFO] Chroma DB is empty. Building index...")

        loader = DocumentLoader()
        docs   = loader.load_folder(DATA_DIR)

        chunker = FixedChunker(chunk_size=CHUNK_SIZE, overlap=OVERLAP)
        chunks  = chunker.chunk_many(docs)

        store.store(chunks)
        print("[INFO] Indexing complete.")
    else:
        print(f"\n[INFO] Existing Chroma DB found with {store.db._collection.count()} stored chunks.")

    # Load prompt builder once
    builder = PromptBuilder(max_context_chars=2000)

    # Load LLM once
    llm = LLMClient(model_name="Qwen/Qwen3-1.7B", temperature=0.0)

    print("=== RAG READY ===\n")
    return store, builder, llm


# Run startup once
store, builder, llm = setup_rag()







 RAG STARTUP
  [loading model] all-MiniLM-L6-v2 ...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  [chroma] collection 'rag' — 620 docs stored

[INFO] Existing Chroma DB found with 620 stored chunks.
  [loading model] Qwen/Qwen3-1.7B ...


Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


=== RAG READY ===



In [42]:
# Function for the Retrieval and Filtering

def retrieve_hits(question: str, top_k: int):
    retriever = Retriever(store=store, top_k=top_k)
    hits = retriever.retrieve(question)
    return hits


def filter_hits_by_score(hits: list[dict], min_score: float):
    return [hit for hit in hits if hit["score"] >= min_score]

In [49]:
# Funtion to build source table showing filename, chunk id, score, and preview.
def format_sources_table(hits: list[dict]) -> pd.DataFrame:
   
    rows = []

    for hit in hits:
        source_path = hit["metadata"].get("source", "Unknown")
        filename    = Path(source_path).name
        chunk_id    = hit["metadata"].get("chunk_id", "N/A")
        score       = hit["score"]

        rows.append({
            "From filename": filename,
            "Chunk": chunk_id,
            "Score": round(score, 4)
        })

    if not rows:
        return pd.DataFrame(columns=["From filename", "Chunk", "Score"])

    return pd.DataFrame(rows)


def build_prompt_from_hits(question: str, hits: list[dict]) -> str:
    return builder.build(question, hits)


In [50]:

# Main Function 'the RAG pipeline 'for Gradio

def run_rag_pipeline(question, top_k, min_score):

    # Basic validation
    if not question or not question.strip():
        empty_df = pd.DataFrame(columns=["For filename", "Chunk", "Score"])
        return "Please enter a question.", "No query provided.", empty_df

    question = question.strip()
    top_k = int(top_k)
    min_score = float(min_score)

    # Retrieving the Top K
    hits = retrieve_hits(question, top_k=top_k)

    # Filtering by threshold
    filtered_hits = filter_hits_by_score(hits, min_score=min_score)

    # if no sources pass threshold, tell user to adjust parameters
    if len(filtered_hits) == 0:
        warning = (
            f"No sources passed the similarity threshold of {min_score:.2f}. "
            f"Try lowering MIN_SCORE or increasing TOP_K."
        )
        empty_df = format_sources_table([])
        return "I don't know based on the retrieved context.", warning, empty_df

    # Building prompt from filtered hits
    prompt = build_prompt_from_hits(question, filtered_hits)

    # Generating the answer
    answer = llm.generate(prompt)

    # Formatting the table display for source
    sources_df = format_sources_table(filtered_hits)

    status = (
        f"{len(filtered_hits)} source(s) passed the threshold "
        f"(TOP_K={top_k}, MIN_SCORE={min_score:.2f})."
    )

    return answer, status, sources_df

# Gradio Interface

with gr.Blocks(title="RAG Question Answering Demo") as demo:
    gr.Markdown(
        """
        Ask a question, then the LLM retrieves relevant chunks, filters them by similarity score,   
        and generates an answer from documents in the dataset set. If answer not found, it replies I don't know.
        """
    )

    with gr.Row():
        with gr.Column(scale=1):
            gr.Markdown("### Retrieval Controls")

            top_k_slider = gr.Slider(
                minimum=1,
                maximum=10,
                step=1,
                value=DEFAULT_TOP_K,
                label="TOP_K"
            )

            min_score_slider = gr.Slider(
                minimum=0.0,
                maximum=1.0,
                step=0.05,
                value=0.4,
                label="MIN_SCORE"
            )

            gr.Markdown(
                """
                **TOP_K** controls how many chunks are retrieved.  
                **MIN_SCORE** filters out weak matches.
                """
            )

        with gr.Column(scale=3):
            question_input = gr.Textbox(
                label="Your Question",
                placeholder="Example: What is RAG?",
                lines=2
            )

            submit_btn = gr.Button("Get Reply")

            answer_output = gr.Textbox(
                label="Generated Answer",
                lines=8
            )

            status_output = gr.Textbox(
                label="Status / Warning",
                lines=2
            )

            sources_output = gr.Dataframe(
                label="Sources",
                headers=["From filename", "Chunk", "Score"],
                datatype=["str", "str", "number"],
                interactive=False
            )

    submit_btn.click(
        fn=run_rag_pipeline,
        inputs=[question_input, top_k_slider, min_score_slider],
        outputs=[answer_output, status_output, sources_output]
    )

In [51]:
demo.launch(share=True)

* Running on local URL:  http://127.0.0.1:7870
* Running on public URL: https://1118a935b2e880188c.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
